# CropSense — 03 Fine-tune Gemma 4 E4B for tool calling (LoRA) → GGUF Q4_0

Trains on `distill_v1/train.jsonl` from notebook 02 and exports a **Q4_0 GGUF** for phones
(llama.cpp on Galaxy S24/S25/S26, CPU/GPU) and your home AMD GPUs (`llama-server`, Vulkan build).
Q4_0 is the format Qualcomm ships for Gemma 4 E4B on Snapdragon (see `colab/AI_PLAN.md`).

**Runtime: L4 GPU** (Runtime → Change runtime type → L4). Turn on **Run in background**, then Run all.
Checkpoints go to Drive; re-running resumes. The runtime **shuts itself down** at the end.

**Budget:** ~3k examples × 2 epochs ≈ 1–2 h training + ~30 min export on L4 ≈ **5–10 compute units**.

Notes (checked 2026-11-20 against Unsloth's Gemma 4 docs and notebook):
- Unsloth's GGUF export supports only Q8_0 / BF16 / F16 for Gemma 4, so we export **F16** and convert to
  **Q4_0** with llama.cpp's `llama-quantize`.
- Gemma 4's tool-call format wasn't confirmed in the docs. `TOOL_FORMAT='auto'` uses the chat template's
  native tool support if it renders tools, otherwise a plain `<tool_call>{json}</tool_call>` format
  (described in the system prompt). The chosen format is saved to `tool_format.txt`; the app must use the same one.
- Not run yet: expect small fixes on the first run.

In [ ]:
BASE_MODEL  = 'unsloth/gemma-4-E4B-it'
RUN_NAME    = 'sahayak_gemma4_e4b_v1'
DATA_DIR    = '/content/drive/MyDrive/cropsense/distill_v1'
OUT_ROOT    = f'/content/drive/MyDrive/cropsense/models/{RUN_NAME}'
MAX_LEN     = 4096
EPOCHS      = 2
LORA_R      = 16
QUANT       = 'Q4_0'      # final GGUF type, made by llama-quantize from the F16 export
TOOL_FORMAT = 'auto'      # 'auto' | 'native' | 'plain'
SHUTDOWN_AT_END = True

In [ ]:
!pip -q install unsloth
# Pins from Unsloth's official Gemma 4 E4B notebook (2026-04). Remove if `pip install unsloth` already resolves them.
!pip -q install "transformers==5.5.0" "tokenizers>=0.22.0,<=0.23.0" "huggingface_hub>=1.5.0,<2.0" "torchao>=0.16.0" "datasets==4.3.0"
from google.colab import drive; drive.mount('/content/drive')
import json, os, glob, re, subprocess, torch
os.makedirs(OUT_ROOT, exist_ok=True)
from unsloth import FastModel
from unsloth.chat_templates import get_chat_template
model, tok = FastModel.from_pretrained(model_name=BASE_MODEL, dtype=None, max_seq_length=MAX_LEN,
                                       load_in_4bit=True, full_finetuning=False)
model = FastModel.get_peft_model(model, finetune_vision_layers=False, finetune_language_layers=True,
    finetune_attention_modules=True, finetune_mlp_modules=True,
    r=LORA_R, lora_alpha=LORA_R, lora_dropout=0, bias='none', random_state=3407)
tok = get_chat_template(tok, chat_template='gemma-4')   # non-thinking template for small Gemma 4 models

def n_tokens(text):
    ids = tok(text=text)['input_ids']
    return len(ids[0]) if ids and isinstance(ids[0], list) else len(ids)

## Data → chat text (native tool template if it works, otherwise plain `<tool_call>` format)

In [ ]:
from datasets import Dataset
TOOLS = json.load(open(f'{DATA_DIR}/tools.json'))
TOOL_NAMES = [t['function']['name'] for t in TOOLS]
def load(name): return [json.loads(l) for l in open(f'{DATA_DIR}/{name}.jsonl')]
train_rows, val_rows = load('train'), load('val')

PLAIN_TOOL_PROMPT = ("\n\nYou can call these tools:\n" + json.dumps(TOOLS, ensure_ascii=False) +
    "\nTo call a tool, reply with ONLY: <tool_call>{\"name\": \"tool_name\", \"arguments\": {...}}</tool_call> "
    "(one per line for several). Tool results come back as <tool_response>...</tool_response>. "
    "When you have what you need, answer the farmer normally.")

def to_plain(msgs):
    out = []
    for m in msgs:
        if m['role'] == 'system':
            out.append({'role': 'system', 'content': m['content'] + PLAIN_TOOL_PROMPT})
        elif m['role'] == 'assistant' and m.get('tool_calls'):
            out.append({'role': 'assistant', 'content': '\n'.join('<tool_call>' + json.dumps(
                {'name': c['function']['name'], 'arguments': c['function']['arguments']}, ensure_ascii=False) + '</tool_call>'
                for c in m['tool_calls'])})
        elif m['role'] == 'tool':
            block = f"<tool_response name=\"{m['name']}\">{m['content']}</tool_response>"
            if out and out[-1]['role'] == 'user' and out[-1]['content'].startswith('<tool_response'):
                out[-1]['content'] += '\n' + block
            else:
                out.append({'role': 'user', 'content': block})
        else:
            out.append({'role': m['role'], 'content': m['content']})
    return out

def native_works():
    sample = next((r for r in train_rows if any(m.get('tool_calls') for m in r['messages'])), None)
    if sample is None: return False
    try:
        text = tok.apply_chat_template(sample['messages'], tools=TOOLS, tokenize=False)
    except Exception as e:
        print('native tool template failed:', e); return False
    called = next(m for m in sample['messages'] if m.get('tool_calls'))['tool_calls'][0]['function']['name']
    tool_text = next(m for m in sample['messages'] if m['role'] == 'tool')['content'][:40]
    return TOOL_NAMES[-1] in text and text.count(called) >= 2 and tool_text in text  # tools listed + call + result rendered

fmt = TOOL_FORMAT if TOOL_FORMAT != 'auto' else ('native' if native_works() else 'plain')
print('tool format:', fmt); open(f'{OUT_ROOT}/tool_format.txt', 'w').write(fmt)

def render(msgs, gen=False):
    if fmt == 'native':
        t = tok.apply_chat_template(msgs, tools=TOOLS, tokenize=False, add_generation_prompt=gen)
    else:
        t = tok.apply_chat_template(to_plain(msgs), tokenize=False, add_generation_prompt=gen)
    return t if gen else t.removeprefix('<bos>')   # the tokenizer adds <bos> again during training

texts = [render(r['messages']) for r in train_rows]
lens = [n_tokens(t) for t in texts[:200]]
print(f'{len(texts)} train examples; sample token lengths: max {max(lens)}, mean {sum(lens)//len(lens)}')
texts = [t for t in texts if n_tokens(t) <= MAX_LEN]
train_ds = Dataset.from_dict({'text': texts})
print(texts[0][:2000])

## Train (loss only on the model turns: tool calls + answers)

In [ ]:
from trl import SFTTrainer, SFTConfig
from unsloth.chat_templates import train_on_responses_only
trainer = SFTTrainer(model=model, tokenizer=tok, train_dataset=train_ds, args=SFTConfig(
    dataset_text_field='text', max_seq_length=MAX_LEN, per_device_train_batch_size=2, gradient_accumulation_steps=8,
    num_train_epochs=EPOCHS, learning_rate=2e-4, warmup_ratio=0.05, lr_scheduler_type='linear', weight_decay=0.001,
    optim='adamw_8bit', logging_steps=10, save_steps=100, save_total_limit=2, output_dir=f'{OUT_ROOT}/checkpoints',
    bf16=torch.cuda.is_bf16_supported(), fp16=not torch.cuda.is_bf16_supported(), report_to='none', seed=3407))
trainer = train_on_responses_only(trainer, instruction_part='<|turn>user\n', response_part='<|turn>model\n')

# Sanity check: the trained (unmasked) part of one example should be only model turns.
ex = trainer.train_dataset[0]
kept = [t for t, l in zip(ex['input_ids'], ex['labels']) if l != -100]
print('TRAINED ON:', tok.decode(kept)[:800])

resume = bool(glob.glob(f'{OUT_ROOT}/checkpoints/checkpoint-*'))
print('resuming' if resume else 'fresh start')
trainer.train(resume_from_checkpoint=resume)
model.save_pretrained(f'{OUT_ROOT}/lora'); tok.save_pretrained(f'{OUT_ROOT}/lora')

## Quick eval on held-out questions
Tool-selection accuracy: does the student call the same first tool as the teacher (or none when the teacher didn't)?
Unsloth warns that Gemma 4 E2B/E4B outputs can go wrong after training with gradient checkpointing, so read the
printed answers, not just the score.

In [ ]:
FastModel.for_inference(model)
def first_call(row):
    for m in row['messages']:
        if m['role'] == 'assistant' and m.get('tool_calls'): return m['tool_calls'][0]['function']['name']
        if m['role'] == 'assistant': return None

def called_tool(out):
    if fmt == 'plain':
        m = re.search(r'<tool_call>\s*(\{.*?\})\s*</tool_call>', out, re.S)
        if not m: return None, True
        try: return json.loads(m.group(1))['name'], True
        except Exception: return 'BAD_JSON', False
    hits = [(out.find(n), n) for n in TOOL_NAMES if n in out]   # native format: first tool name the model emits
    return (min(hits)[1] if hits else None), True

ok = good_json = n = 0
for row in val_rows[:100]:
    prompt = render(row['messages'][:2], gen=True)
    ids = tok(text=[prompt], return_tensors='pt', add_special_tokens=False).to('cuda')
    gen = model.generate(**ids, max_new_tokens=256, do_sample=False)
    out = tok.decode(gen[0][ids['input_ids'].shape[1]:], skip_special_tokens=False)
    name, parsed = called_tool(out)
    n += 1; ok += (name == first_call(row)); good_json += parsed
    if n <= 5: print('Q:', row['messages'][1]['content'], '\n→', out[:400], '\n')
print(f'tool selection accuracy: {ok}/{n}  | well-formed calls: {good_json}/{n}')

## Export GGUF: F16 from Unsloth → Q4_0 with llama.cpp

In [ ]:
model.save_pretrained_gguf(f'{OUT_ROOT}/gguf', tok, quantization_method='f16')
f16 = sorted(glob.glob(f'{OUT_ROOT}/gguf/**/*.gguf', recursive=True) + glob.glob(f'{OUT_ROOT}/gguf*.gguf'), key=os.path.getsize)[-1]
print('F16 file:', f16)

if not os.path.exists('/content/llama.cpp/build/bin/llama-quantize'):
    subprocess.run('git clone --depth 1 https://github.com/ggml-org/llama.cpp /content/llama.cpp && '
                   'cmake -S /content/llama.cpp -B /content/llama.cpp/build -DGGML_CUDA=OFF -DLLAMA_CURL=OFF && '
                   'cmake --build /content/llama.cpp/build --target llama-quantize -j 8', shell=True, check=True)
q_out = f'{OUT_ROOT}/{RUN_NAME}-{QUANT}.gguf'
subprocess.run(['/content/llama.cpp/build/bin/llama-quantize', f16, q_out, QUANT], check=True)
print(f'{q_out}: {os.path.getsize(q_out)/1e9:.2f} GB')
print('Home box test:  llama-server -m', os.path.basename(q_out), '--jinja -ngl 99 --port 8081')

In [ ]:
if SHUTDOWN_AT_END:
    from google.colab import runtime; runtime.unassign()